# 📄 Universal Document Restoration Tool

Restores scanned, aged, stained or damaged documents to **clean black type on white paper** —
exactly as if the document had been retyped from scratch.

**How it works:**
1. You supply a scanned document image (JPEG / PNG / TIFF / BMP / WebP)
2. The Claude Vision API reads every word, number and table in the image
3. The extracted content is typeset into a brand-new, clean PDF using ReportLab
4. Output is indistinguishable from a freshly printed document

This replicates the exact process used to restore the British West African
Meteorological Services record: Claude read the degraded scan and produced
a perfectly clean PDF — no image processing required.

**Requirements:** An Anthropic API key (set as `ANTHROPIC_API_KEY` environment variable or entered in Step 2).

---
## Step 1 — Install Dependencies

In [ ]:
!pip install anthropic reportlab Pillow tqdm

---
## Step 2 — Imports & API Key

In [ ]:
import os
import base64
import io
from pathlib import Path

import anthropic
from PIL import Image
from reportlab.lib.pagesizes import A4, letter
from reportlab.lib.units import mm
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, HRFlowable
)
from tqdm import tqdm
from IPython.display import display, HTML, IFrame

# ── API Key ───────────────────────────────────────────────────────────────────
# Option A: set environment variable before launching Jupyter:
#   export ANTHROPIC_API_KEY="sk-ant-..."
# Option B: paste your key directly here (never commit to git):
API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")

if not API_KEY:
    API_KEY = input("INSERT KEY HERE").strip()

client = anthropic.Anthropic(api_key=API_KEY)
print("✅ Anthropic client ready.")

---
## Step 3 — Configuration

Set your input image path and output PDF path here.

In [ ]:
# ── Edit these ────────────────────────────────────────────────────────────────
INPUT_IMAGE  = r"C:\Users\            # ← path to your scanned document image
OUTPUT_PDF   = "restored_output.pdf"  # ← where the clean PDF will be saved
PAGE_SIZE    = A4                     # A4 or letter
FONT_SIZE    = 9                      # base font size for body text
MARGINS_MM   = 15                     # page margins in mm

# Optional: add context about the document type to improve OCR accuracy
# Leave as empty string for auto-detection
DOCUMENT_CONTEXT = ""
# Examples:
# DOCUMENT_CONTEXT = "This is a meteorological observation log from the 1950s."
# DOCUMENT_CONTEXT = "This is a handwritten financial ledger."
# DOCUMENT_CONTEXT = "This is a medical record with handwritten physician notes."
# ─────────────────────────────────────────────────────────────────────────────

print(f"Input : {INPUT_IMAGE}")
print(f"Output: {OUTPUT_PDF}")

---
## Step 4 — OCR: Claude Vision reads the document

Claude reads the image and returns every piece of content in structured plain text,
preserving headings, tables, labels, values and handwritten notes.
It produces the same quality output as transcribing manually — because that is
exactly what it does.

In [ ]:
def image_to_base64(path: str) -> tuple:
    """
    Load an image file and return (base64_data, media_type).
    Converts TIFF / BMP to JPEG for API compatibility.
    """
    p = Path(path)
    suffix = p.suffix.lower()
    media_map = {
        ".jpg":  "image/jpeg",
        ".jpeg": "image/jpeg",
        ".png":  "image/png",
        ".gif":  "image/gif",
        ".webp": "image/webp",
    }
    if suffix in media_map:
        with open(path, "rb") as f:
            return base64.standard_b64encode(f.read()).decode(), media_map[suffix]
    else:
        # Convert unsupported formats to JPEG
        img = Image.open(path).convert("RGB")
        buf = io.BytesIO()
        img.save(buf, format="JPEG", quality=95)
        return base64.standard_b64encode(buf.getvalue()).decode(), "image/jpeg"


BASE_OCR_PROMPT = """\
You are an expert document transcription assistant.

Carefully read EVERY piece of text in this scanned document image, including:
- All printed headings, section titles, labels and field names
- All handwritten entries, numbers, dates and signatures
- All table column headers and every cell value
- All footnotes, stamps, and marginal notes

Output the COMPLETE transcription using ONLY these prefixed line types:

  HEADING: <main title text>
  SUBHEADING: <secondary title text>
  SECTION: <section divider label, e.g. "EXPOSURE OF INSTRUMENTS">
  FIELD: <label> | <value>
  TABLE_HEADER: <col1> | <col2> | <col3> ...
  TABLE_ROW: <val1> | <val2> | <val3> ...
  TEXT: <any free-form paragraph or sentence>
  NOTE: <footnote, marginal note, or asterisked remark>

Rules:
- Output EVERY visible character. Do not skip or summarise anything.
- Empty table cells: use a single space between pipes, e.g.  |   |  
- Illegible text: write [illegible] at that position.
- Preserve reading order: top to bottom, left to right.
- Output ONLY the transcription — no commentary, no apology, no explanation.
"""


def ocr_document(image_path: str, context: str = "") -> str:
    """
    Send a document image to Claude Vision and return the full structured transcription.

    Parameters
    ----------
    image_path : str
        Path to the scanned document image.
    context : str
        Optional description of the document type to improve accuracy.
    """
    print(f"📤 Sending '{image_path}' to Claude Vision API...")
    b64, media_type = image_to_base64(image_path)

    prompt = BASE_OCR_PROMPT
    if context:
        prompt += f"\nDocument context: {context}\n"

    response = client.messages.create(
        model="claude-opus-4-5",
        max_tokens=8192,
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": media_type,
                        "data": b64,
                    }
                },
                {"type": "text", "text": prompt}
            ]
        }]
    )

    transcription = response.content[0].text
    print(f"✅ Transcription received: {len(transcription):,} characters")
    print(f"   Input tokens:  {response.usage.input_tokens:,}")
    print(f"   Output tokens: {response.usage.output_tokens:,}")
    return transcription


print("✅ OCR function defined.")

---
## Step 5 — PDF Typesetter

Parses the structured transcription and renders it as a clean, typeset PDF.
Headings are centred and bold, fields are label: value pairs, tables get
proper grid lines, and notes appear in italic.

In [ ]:
def build_pdf(transcription: str,
              output_path: str,
              page_size=A4,
              font_size: int = 9,
              margins_mm: int = 15) -> str:
    """
    Parse the structured transcription and render a clean typeset PDF.

    Recognised line prefixes
    ------------------------
    HEADING        → centred bold large title
    SUBHEADING     → centred bold medium title
    SECTION        → centred bold with horizontal rules above/below
    FIELD          → 'label:  value' (pipe-separated in transcription)
    TABLE_HEADER   → bold grey header row in a grid table
    TABLE_ROW      → data row in a grid table
    TEXT           → normal paragraph
    NOTE           → italic small footnote
    """
    m   = margins_mm * mm
    doc = SimpleDocTemplate(
        output_path,
        pagesize=page_size,
        leftMargin=m, rightMargin=m,
        topMargin=m,  bottomMargin=m,
    )

    # ── Text styles ───────────────────────────────────────────────────────────
    styles = {
        "heading": ParagraphStyle(
            "H1", fontName="Helvetica-Bold",
            fontSize=font_size + 4, spaceAfter=4, spaceBefore=6, alignment=1
        ),
        "subheading": ParagraphStyle(
            "H2", fontName="Helvetica-Bold",
            fontSize=font_size + 2, spaceAfter=3, spaceBefore=4, alignment=1
        ),
        "section": ParagraphStyle(
            "SEC", fontName="Helvetica-Bold",
            fontSize=font_size + 1, spaceAfter=2, spaceBefore=2, alignment=1
        ),
        "field": ParagraphStyle(
            "FLD", fontName="Helvetica",
            fontSize=font_size, spaceAfter=1, spaceBefore=1
        ),
        "text": ParagraphStyle(
            "TXT", fontName="Helvetica",
            fontSize=font_size, spaceAfter=3, spaceBefore=2
        ),
        "note": ParagraphStyle(
            "NOTE", fontName="Helvetica-Oblique",
            fontSize=font_size - 1, spaceAfter=2, spaceBefore=2,
            textColor=colors.HexColor("#444444")
        ),
    }
    th_style = ParagraphStyle("TH", fontName="Helvetica-Bold",
                              fontSize=font_size - 1, alignment=1)
    td_style = ParagraphStyle("TD", fontName="Helvetica",
                              fontSize=font_size - 1, alignment=1)

    # ── State ─────────────────────────────────────────────────────────────────
    story             = []
    pending_rows      = []   # list of ("header"|"row", [cell, ...])
    usable_w          = page_size[0] - 2 * m

    def flush_table():
        """Emit any accumulated table rows as a ReportLab Table."""
        if not pending_rows:
            return
        ncols = max(len(r) for _, r in pending_rows)
        data  = []
        is_hdr = []
        for kind, cells in pending_rows:
            # Pad / trim to ncols
            padded = (cells + [""] * ncols)[:ncols]
            sty    = th_style if kind == "header" else td_style
            data.append([Paragraph(str(c), sty) for c in padded])
            is_hdr.append(kind == "header")

        col_w = usable_w / ncols
        tbl   = Table(data, colWidths=[col_w] * ncols, repeatRows=1)
        ts    = TableStyle([
            ("GRID",           (0,0), (-1,-1), 0.4,  colors.black),
            ("FONTSIZE",       (0,0), (-1,-1), font_size - 1),
            ("TOPPADDING",     (0,0), (-1,-1), 2),
            ("BOTTOMPADDING",  (0,0), (-1,-1), 2),
            ("LEFTPADDING",    (0,0), (-1,-1), 3),
            ("RIGHTPADDING",   (0,0), (-1,-1), 3),
            ("VALIGN",         (0,0), (-1,-1), "MIDDLE"),
            ("ALIGN",          (0,0), (-1,-1), "CENTER"),
        ])
        for i, h in enumerate(is_hdr):
            if h:
                ts.add("BACKGROUND", (0,i), (-1,i), colors.HexColor("#D8D8D8"))
                ts.add("FONTNAME",   (0,i), (-1,i), "Helvetica-Bold")
        tbl.setStyle(ts)
        story.append(tbl)
        story.append(Spacer(1, 4))
        pending_rows.clear()

    def split_pipe(s: str):
        return [c.strip() for c in s.split("|")]

    # ── Parse lines ───────────────────────────────────────────────────────────
    for raw in transcription.splitlines():
        line = raw.strip()
        if not line:
            continue

        if ":" not in line:
            flush_table()
            story.append(Paragraph(line, styles["text"]))
            continue

        prefix, _, rest = line.partition(":")
        key  = prefix.strip().upper()
        rest = rest.strip()

        if key == "HEADING":
            flush_table()
            story.append(Paragraph(rest, styles["heading"]))

        elif key == "SUBHEADING":
            flush_table()
            story.append(Paragraph(rest, styles["subheading"]))

        elif key == "SECTION":
            flush_table()
            story.append(Spacer(1, 5))
            story.append(HRFlowable(width="100%", thickness=0.6, color=colors.black))
            story.append(Paragraph(rest, styles["section"]))
            story.append(HRFlowable(width="100%", thickness=0.6, color=colors.black))
            story.append(Spacer(1, 3))

        elif key == "FIELD":
            flush_table()
            parts = split_pipe(rest)
            if len(parts) >= 2:
                label = parts[0]
                value = "   ".join(parts[1:])
                story.append(Paragraph(f"<b>{label}:</b>  {value}",
                                       styles["field"]))
            else:
                story.append(Paragraph(rest, styles["field"]))

        elif key == "TABLE_HEADER":
            pending_rows.append(("header", split_pipe(rest)))

        elif key == "TABLE_ROW":
            pending_rows.append(("row", split_pipe(rest)))

        elif key == "NOTE":
            flush_table()
            story.append(Paragraph(f"* {rest}", styles["note"]))

        elif key == "TEXT":
            flush_table()
            story.append(Paragraph(rest, styles["text"]))

        else:
            # Unknown prefix — render as plain text
            flush_table()
            story.append(Paragraph(line, styles["text"]))

    flush_table()  # flush any trailing table

    doc.build(story)
    print(f"✅ Clean PDF saved → {output_path}")
    return output_path


print("✅ PDF typesetter defined.")

---
## Step 6 — Run: OCR → Clean PDF

This single cell reads the scan and produces the restored PDF.

In [ ]:
# ── 1. OCR ────────────────────────────────────────────────────────────────────
transcription = ocr_document(INPUT_IMAGE, context=DOCUMENT_CONTEXT)

# ── 2. Preview raw transcription ─────────────────────────────────────────────
print("\n" + "─" * 70)
print("RAW TRANSCRIPTION (first 3000 chars):")
print("─" * 70)
print(transcription[:3000])
if len(transcription) > 3000:
    print(f"\n... [{len(transcription) - 3000:,} more characters]")

# ── 3. Typeset into clean PDF ─────────────────────────────────────────────────
build_pdf(
    transcription,
    OUTPUT_PDF,
    page_size  = PAGE_SIZE,
    font_size  = FONT_SIZE,
    margins_mm = MARGINS_MM,
)

---
## Step 7 — Preview the PDF inline

In [ ]:
IFrame(OUTPUT_PDF, width=900, height=1100)

---
## Step 8 — Edit & Re-typeset (optional)

If the OCR missed or misread anything, edit the `transcription` string here
and re-run `build_pdf()` — no need to pay for another API call.

In [ ]:
# Make corrections to the transcription:
# corrected = transcription.replace("wrong value", "correct value")
#
# Re-build the PDF with the corrected text:
# build_pdf(corrected, "corrected_output.pdf")
#
# Or edit directly in a text editor:
# with open("transcription.txt", "w") as f:
#     f.write(transcription)
# # ... edit transcription.txt ...
# with open("transcription.txt") as f:
#     edited = f.read()
# build_pdf(edited, OUTPUT_PDF)

print("Edit the lines above to correct any OCR errors, then run this cell.")

---
## Step 9 — Batch: restore a whole folder of documents

In [ ]:
SUPPORTED_EXT = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp"}


def restore_folder(input_folder: str,
                   output_folder: str,
                   context: str = "") -> list:
    """
    OCR + typeset every document image in input_folder.
    Saves one clean PDF per image into output_folder.

    Parameters
    ----------
    input_folder  : folder containing scanned images
    output_folder : folder to write restored PDFs into
    context       : optional document-type hint (applied to all images)
    """
    in_dir  = Path(input_folder)
    out_dir = Path(output_folder)
    out_dir.mkdir(parents=True, exist_ok=True)

    files = sorted(f for f in in_dir.iterdir()
                   if f.suffix.lower() in SUPPORTED_EXT)
    print(f"Found {len(files)} document image(s) in '{input_folder}'")

    results = []
    for f in tqdm(files, desc="Restoring"):
        out_pdf = out_dir / (f.stem + "_restored.pdf")
        try:
            t = ocr_document(str(f), context=context)
            build_pdf(t, str(out_pdf),
                      page_size=PAGE_SIZE,
                      font_size=FONT_SIZE,
                      margins_mm=MARGINS_MM)
            results.append({"source": str(f), "output": str(out_pdf), "ok": True})
        except Exception as e:
            print(f"  ⚠️  {f.name}: {e}")
            results.append({"source": str(f), "output": None, "ok": False, "error": str(e)})

    ok = sum(1 for r in results if r["ok"])
    print(f"\n✅ Done. {ok}/{len(files)} restored → '{output_folder}/'")
    return results


# ── Uncomment to run batch mode: ──────────────────────────────────────────────
# results = restore_folder(
#     input_folder  = "scans_folder/",
#     output_folder = "restored_output/",
#     context       = "These are meteorological observation records from the 1950s."
# )

print("✅ Batch function ready. Uncomment the last block to run.")

---
## Appendix — Tuning the OCR prompt

Providing `DOCUMENT_CONTEXT` is the easiest way to improve accuracy for
specialist documents. You can also append directly to `BASE_OCR_PROMPT`:

```python
# Meteorological / scientific logs:
BASE_OCR_PROMPT += """
This is a meteorological observation record. Pay special attention to:
- Station name, latitude, longitude, date and year
- All instrument names, serial numbers and correction values
- Every numeric reading in every table cell, no matter how faint
"""

# Handwritten ledgers:
BASE_OCR_PROMPT += """
This is a handwritten financial ledger.
Mark crossed-out entries as [crossed out: <value>].
Use TABLE_ROW for each ledger line.
"""

# Medical records:
BASE_OCR_PROMPT += """
This is a medical record. Preserve all abbreviations exactly.
Include all dates, dosages, and physician signatures.
"""

# Legal documents:
BASE_OCR_PROMPT += """
This is a legal document. Preserve all case numbers, party names,
dates, and article/clause numbering exactly as written.
"""
```

The more context Claude has about the document type, the more accurately
it will resolve ambiguous handwriting and specialist abbreviations.